# 04 — Обратная связь u=Kx

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import ensure_dir, set_seed, save_dataframe, save_json
from src.systems import cross_coupled_uncontrolled
from src.simulation import simulate_batch, uncertain_dynamics
from src.plots import set_plot_style, save_figure, plot_phase_trajectories, plot_residual_histogram, plot_lyapunov_contours
from src.basis import get_basis
from src.identification import fit_identified_model, predict_vector_field, rmse
from src.uncertainty import compute_residuals, residual_norms, estimate_epsilon, compute_bounding_box, bounded_disturbance
from src.lyapunov import numerical_jacobian, solve_lyapunov, evaluate_lyapunov_grid
from src.control import candidate_gains, closed_loop_jacobian


In [ ]:
set_plot_style()
df = pd.read_csv('data/processed/cross_coupled_dataset.csv')
X = df[['x1','x2']].to_numpy(); Xdot = df[['xdot1','xdot2']].to_numpy()
basis_fn = get_basis('quadratic_with_constant')
res = fit_identified_model(X, Xdot, basis_fn=basis_fn, basis_name='quadratic_with_constant')

A = numerical_jacobian(lambda x: predict_vector_field(x[None, :], basis_fn, res.coefficients)[0], np.zeros(2))
B = np.array([[0.0], [1.0]])
epsilon = estimate_epsilon(residual_norms(compute_residuals(Xdot, predict_vector_field(X, basis_fn, res.coefficients))), q=0.95)
initials = np.array([[-1.0,-0.8],[-0.8,1.0],[0.8,-1.0],[1.1,0.9]])
t_eval = np.linspace(0, 12, 500)

plt.figure()
for name, K in candidate_gains().items():
    def cl_dyn(t, x, K=K):
        u = K @ x
        return cross_coupled_uncontrolled(t, x) + (B @ u).reshape(-1) + bounded_disturbance(t, epsilon)
    traj = simulate_batch(cl_dyn, initials, (0, 12), t_eval)
    for idx, (_, st) in enumerate(traj):
        plt.plot(st[:,0], st[:,1], lw=1.0, label=name if idx == 0 else None)

plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Controlled uncertain trajectories for candidate K'); plt.legend()
save_figure('results/figures/uncertain_controlled_comparison.png'); plt.show()

metrics = {}
for name, K in candidate_gains().items():
    Acl = closed_loop_jacobian(A, B, K)
    eig = np.linalg.eigvals(Acl)
    metrics[name] = {'K': K.tolist(), 'eig_real_parts': np.real(eig).tolist()}
save_json(metrics, 'results/metrics/closed_loop_comparison.json')
